# Tune the cuts and match filters on known dwarf galaxies and globular clusters

A known object at the distance modulus of interest is a real old, metal-poor population: the signal of a
selection (cuts, colour or magnitude ranges, a match filter in some bands) is its excess of stars on the object
over the field around it. Objects from `catalogs.overlapped_objects` (LVDB), regions and SNR in
`analysis.gc_dwarf_optimization`, plots in `plotting.plot_gc_dwarf_optimization`.

Every selection is a boolean array over the rows of the catalog: combine them with `&` and `|`, with any cut on
the catalog columns. The catalog magnitudes are already dereddened.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from streameucliddes.analysis.catalog_preparation import catalog_match_filter_mask, prepare_match_filter_catalog
from streameucliddes.analysis.cmd_match_filter import (
    isochrone_cmd_track,
    match_filter_cmd_columns,
    on_off_hess,
)
from streameucliddes.analysis.gc_dwarf_optimization import (
    compare_selections,
    object_field_radius_deg,
    object_radec_box,
    object_regions,
    selection_snr,
)
from streameucliddes.catalogs.euclid_catalog import default_star_cuts, get_euclid_stars
from streameucliddes.catalogs.isochrone_parameters import isochrone_from_feh
from streameucliddes.catalogs.overlapped_objects import find_object, make_object, objects_near_distance_modulus
from streameucliddes.plotting.plot_cmd import plot_hess, plot_isochrone, plot_match_filter_polygon
from streameucliddes.plotting.plot_gc_dwarf_optimization import (
    plot_object_cmd,
    plot_object_hess,
    plot_object_map,
    plot_selection_comparison,
)

## 1. Known objects near a distance modulus

In [ ]:
DM_OF_INTEREST = 17.0
MAX_DM_OFFSET = 1.0
CATALOG = "euclid_dr1"  # Euclid DR1 stars, Euclid bands only

# in_catalog: sources of the catalog at the centre of the object (HATS point map, no star read)
objects = objects_near_distance_modulus(DM_OF_INTEREST, MAX_DM_OFFSET, object_types=("dwarfs", "gc"), catalog=CATALOG)
objects[objects["in_catalog"]][["type", "key", "name", "radius_arcmin", "ellipticity", "distance_modulus", "dm_offset",
                                "feh", "n_sources_at_centre"]]

## 2. One object and its field

The on region is an ellipse (or an elliptical annulus) between `ON_RADII` half-light radii, the off region an
elliptical annulus between `OFF_RADII`. For a globular cluster, start the on annulus beyond its core, where the
catalog misses stars (crowding), and the off annulus beyond its stars: e.g. NGC 1851, `ON_RADII = (4, 15)`,
`OFF_RADII = (30, 40)`.

In [ ]:
# Objects made by hand (e.g. new discoveries, not in the LVDB): make_object(name, ra, dec, half-light radius in arcmin,
# ...), unknown values NaN (age: the default one)
HANDMADE_OBJECTS = {
    # Gatto et al. (Euclid DR1), Tables 1 and 2: dm 16.5 +- 0.3, [Fe/H] ~ -2.2, M_V = +2.0, so V = M_V + dm
    "Ursa Minor II": make_object("Ursa Minor II", 196.1542, 75.5724, 2.25, ellipticity=0.20, position_angle=162.0,
                                 distance_modulus=16.5, feh=-2.2, apparent_magnitude_v=2.0 + 16.5),
}
OBJECT_NAME = "Reticulum II"  # an LVDB object (find_object) or one of HANDMADE_OBJECTS, e.g. "Ursa Minor II"
#OBJECT_NAME = "Ursa Minor II"
ON_RADII = (0.0, 2.0)  # half-light radii
OFF_RADII = (5.0, 10.0)
# ugali isochrone model of the match filters: "Marigo2017" (PARSEC, default) or "Dotter2016" (MIST v2.5 for Euclid);
# [Fe/H] converted to the Z of the model
ISOCHRONE_MODEL = "Marigo2017"

obj = HANDMADE_OBJECTS[OBJECT_NAME] if OBJECT_NAME in HANDMADE_OBJECTS else find_object(OBJECT_NAME)
# LVDB age and [Fe/H] (as Z); the default age (13 Gyr) when LVDB has none
isochrone = isochrone_from_feh(obj["age_gyr"], feh=obj["feh"], isochrone_model=ISOCHRONE_MODEL)
distance_modulus = obj["distance_modulus"]
print(obj[["name", "radius_arcmin", "ellipticity", "position_angle", "distance_modulus", "feh", "age_gyr"]].to_dict())
print("isochrone:", isochrone)

In [ ]:
ra_range, dec_range = object_radec_box(obj, object_field_radius_deg(obj, OFF_RADII))

STAR_SELECTION = "morphology_cuts"  # or "xgboost_classifier" (see default_star_cuts)

catalog = get_euclid_stars(CATALOG, cuts=default_star_cuts(CATALOG, STAR_SELECTION), ra_range=ra_range,
                           dec_range=dec_range)
print([column for column in catalog.columns if column.endswith("_obs")])

In [ ]:
STAR_SELECTION_gatto = "gatto_stars_dr1"
catalog_gatto = get_euclid_stars(CATALOG, cuts=default_star_cuts(CATALOG, STAR_SELECTION_gatto), ra_range=ra_range,dec_range=dec_range)

In [ ]:
# Morphology cuts with a loose POINT_LIKE_PROB threshold: min_point_like_prob replaces the 0.5 of "morphology_cuts"
catalog_low_pointlikeprobe = get_euclid_stars(
    CATALOG, cuts=default_star_cuts(CATALOG, "morphology_cuts", min_point_like_prob=0.05),
    ra_range=ra_range, dec_range=dec_range,
)

In [ ]:
regions = object_regions(catalog, obj, on_radii=ON_RADII, off_radii=OFF_RADII)
print(f"on: {regions['on'].sum()} stars, {regions['on_area_deg2']:.4f} deg2 ({regions['on_area_fraction']:.0%} kept)")
print(f"off: {regions['off'].sum()} stars, {regions['off_area_deg2']:.4f} deg2 ({regions['off_area_fraction']:.0%} kept)")
print(f"{len(regions['masked_objects'])} other objects masked in the field")
fig, ax = plot_object_map(regions, smoothing_deg=0.03)  # all the stars; north up, east left

## S/G: POINT_LIKE_PROB cuts on the Gatto and morphology stars

Stricter `POINT_LIKE_PROB` cuts on two catalogs read with `POINT_LIKE_PROB >= 0.05`: the Gatto stars
(`catalog_gatto`, with the colour box) and the morphology stars (`catalog_low_pointlikeprobe`, with the
`ELLIPTICITY` and `MUMAX_MINUS_MAG` cuts). Reference: the default morphology stars of section 2 (`catalog`,
`POINT_LIKE_PROB >= 0.5`). Each catalog has its own on/off regions, with the same geometry, so their SNRs compare.

In [ ]:
MATCH_FILTER_SG = {"name": "vis_h", "reference_band": ("euclid_dr1", "VIS"), "color_bands": [("euclid_dr1", "H")]}
POINT_LIKE_PROB_CUTS = np.linspace(0.05, 0.95, 10)


def point_like_prob_scan(stars, name):
    """SNR table of the match filter on ``stars``, alone (``name``) and with each POINT_LIKE_PROB cut."""
    stars_regions = object_regions(stars, obj, on_radii=ON_RADII, off_radii=OFF_RADII)
    match_filter = catalog_match_filter_mask(stars, distance_modulus, isochrone, MATCH_FILTER_SG)
    point_like_prob = stars["POINT_LIKE_PROB"].to_numpy()
    selections = {
        name: match_filter,
        **{f"pointlike_prob_{cut:.2f}": match_filter & (point_like_prob > cut) for cut in POINT_LIKE_PROB_CUTS},
    }
    return compare_selections(stars_regions, selections), stars_regions


gatto_table, regions_gatto = point_like_prob_scan(catalog_gatto, "gatto_sep")
euclid_table, regions_low_pointlikeprob = point_like_prob_scan(catalog_low_pointlikeprobe, "euclid_sep_0.05")

# Reference: the default morphology stars of section 2
euclid_snr = selection_snr(regions, catalog_match_filter_mask(catalog, distance_modulus, isochrone, MATCH_FILTER_SG))
print(f"euclid_sep (morphology_cuts): SNR {euclid_snr['snr']:.1f}, {euclid_snr['n_on']} stars on the object")
display(gatto_table.round(2))
display(euclid_table.round(2))

In [ ]:
fig, ax = plot_object_map(regions_gatto, smoothing_deg=0.03)  # all the stars; north up, east left

In [ ]:
fig, ax = plt.subplots(layout="constrained")
gatto_color = "C1"
morph_color = "C0"

ax.axhline(euclid_snr["snr"], color=morph_color, ls="--", label="Morph.")
ax.axhline(gatto_table.loc["gatto_sep", "snr"], color=gatto_color, ls="--", label="Gatto")
names = [f"pointlike_prob_{cut:.2f}" for cut in POINT_LIKE_PROB_CUTS]
ax.plot(POINT_LIKE_PROB_CUTS, gatto_table.loc[names, "snr"], color=gatto_color, marker="o",
        label=r"Gatto POINT_LIKE_PROB $>$ cut")
ax.plot(POINT_LIKE_PROB_CUTS, euclid_table.loc[names, "snr"], color=morph_color, marker="o",
        label=r"Morph POINT_LIKE_PROB $>$ cut")
ax.set_xlabel("POINT_LIKE_PROB cut")
ax.set_ylabel("SNR")
ax.legend()

# Isochrone model

In [ ]:
# On - off Hess diagram (VIS vs VIS - H, stars passing the error cuts of the match filter) of each star selection,
# with the isochrone (solid) and the match filter (dotted) of each model, and the SNR of each match filter: the
# isochrone of the object (LVDB age, else the default one, and [Fe/H] converted to the Z of each model)
ISOCHRONE_MODEL_COLORS = {"Marigo2017": "k", "Dotter2016": "C2"}
mag_cols = match_filter_cmd_columns(MATCH_FILTER_SG)[0]  # (VIS, H)
model_isochrones = {isochrone_model: isochrone_from_feh(obj["age_gyr"], feh=obj["feh"], isochrone_model=isochrone_model)
                    for isochrone_model in ISOCHRONE_MODEL_COLORS}
tracks = {
    isochrone_model: isochrone_cmd_track(distance_modulus, model_isochrone["age"], model_isochrone["metallicity"],
                                         match_filter=MATCH_FILTER_SG, isochrone_model=isochrone_model)
    for isochrone_model, model_isochrone in model_isochrones.items()
}

fig, axes = plt.subplots(1, 1, layout="constrained")
axes = [axes]
for ax, (name, stars, stars_regions) in zip(axes, [
                                                   ("Gatto", catalog_gatto, regions_gatto)]):
    shown = prepare_match_filter_catalog(stars, MATCH_FILTER_SG, as_mask=True)  # its error cuts
    _, _, difference = on_off_hess(stars, mag_cols, shown & stars_regions["on"], stars_regions["on_area_deg2"],
                                   [shown & stars_regions["off"]], [stars_regions["off_area_deg2"]])
    plot_hess(difference, ax=ax, difference=True, smoothing_bins=1.,
              title=f"{name}: on $-$ off, {difference['n_stars']:.0f} excess stars")
    for isochrone_model, color in ISOCHRONE_MODEL_COLORS.items():
        plot_isochrone(ax, tracks[isochrone_model], color=color, label=isochrone_model)
        plot_match_filter_polygon(ax, distance_modulus, model_isochrones[isochrone_model], MATCH_FILTER_SG,
                                  color=color, ls=":", label=f"{isochrone_model} match filter")
    for row, (isochrone_model, color) in enumerate(ISOCHRONE_MODEL_COLORS.items()):  # top right, one line per model
        selected = catalog_match_filter_mask(stars, distance_modulus, model_isochrones[isochrone_model], MATCH_FILTER_SG)
        snr = selection_snr(stars_regions, selected)["snr"]
        ax.text(0.97, 0.97 - 0.06 * row, f"SNR = {snr:.1f}", transform=ax.transAxes, ha="right", va="top", color=color)
    ax.legend(loc="upper left", fontsize="small")
axes[0].set_ylabel("")

In [ ]:
# On - off Hess diagram (VIS vs VIS - H, stars passing the error cuts of the match filter) of each star selection,
# with the isochrone (solid) and the match filter (dotted) of each model, and the SNR of each match filter: the
# isochrone of the object (LVDB age, else the default one, and [Fe/H] converted to the Z of each model)
ISOCHRONE_MODEL_COLORS = {"Marigo2017": "k", "Dotter2016": "C2"}
mag_cols = match_filter_cmd_columns(MATCH_FILTER_SG)[0]  # (VIS, H)
model_isochrones = {isochrone_model: isochrone_from_feh(obj["age_gyr"], feh=obj["feh"], isochrone_model=isochrone_model)
                    for isochrone_model in ISOCHRONE_MODEL_COLORS}
tracks = {
    isochrone_model: isochrone_cmd_track(distance_modulus, model_isochrone["age"], model_isochrone["metallicity"],
                                         match_filter=MATCH_FILTER_SG, isochrone_model=isochrone_model)
    for isochrone_model, model_isochrone in model_isochrones.items()
}

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True, layout="constrained")
for ax, (name, stars, stars_regions) in zip(axes, [("Morphology", catalog, regions),
                                                   ("Gatto", catalog_gatto, regions_gatto)]):
    shown = prepare_match_filter_catalog(stars, MATCH_FILTER_SG, as_mask=True)  # its error cuts
    _, _, difference = on_off_hess(stars, mag_cols, shown & stars_regions["on"], stars_regions["on_area_deg2"],
                                   [shown & stars_regions["off"]], [stars_regions["off_area_deg2"]])
    plot_hess(difference, ax=ax, difference=True, smoothing_bins=1,
              title=f"{name}: on $-$ off, {difference['n_stars']:.0f} excess stars")
    for isochrone_model, color in ISOCHRONE_MODEL_COLORS.items():
        plot_isochrone(ax, tracks[isochrone_model], color=color, label=isochrone_model)
        plot_match_filter_polygon(ax, distance_modulus, model_isochrones[isochrone_model], MATCH_FILTER_SG,
                                  color=color, ls=":", label=f"{isochrone_model} match filter")
    for row, (isochrone_model, color) in enumerate(ISOCHRONE_MODEL_COLORS.items()):  # top right, one line per model
        selected = catalog_match_filter_mask(stars, distance_modulus, model_isochrones[isochrone_model], MATCH_FILTER_SG)
        snr = selection_snr(stars_regions, selected)["snr"]
        ax.text(0.97, 0.97 - 0.06 * row, f"SNR = {snr:.0f}", transform=ax.transAxes, ha="right", va="top", color=color)
    ax.legend(loc="upper left", fontsize="small")
axes[1].set_ylabel("")

In [ ]:
# Width of the match filter: error_multiplier times the photometric error plus color_spread, each (blue side, red side);
# absent: the streamobs defaults, (2, 2) and (0.05, 0.05) mag
WIDTHS = {
    "default": {},
    "color_spread (0.05, 0.1)": {"color_spread": (0.05, 0.1)},  # DES 2018 (Shipp et al.)
    "color_spread (0.1, 0.1)": {"color_spread": (0.1, 0.1)},
    "error_multiplier (1, 1)": {"error_multiplier": (1.0, 1.0)},
    "error_multiplier (3, 3)": {"error_multiplier": (3.0, 3.0)},
    "color_spread (0.1, 0.1) + error_multiplier (3, 3)": {"color_spread": (0.1, 0.1), "error_multiplier": (3.0, 3.0)},
     "color_spread (0.05, 0.15)": {"color_spread": (0.05, 0.15),},
     "color_spread (0.05, 0.15) + error_multiplier (3, 3)": {"color_spread": (0.05, 0.15), "error_multiplier": (3.0, 3.0)}
}
# label: stars, their regions, isochrone (object age and [Fe/H] in each model), marker, colour
isochrone_marigo = isochrone_from_feh(obj["age_gyr"], feh=obj["feh"], isochrone_model="Marigo2017")
isochrone_dotter = isochrone_from_feh(obj["age_gyr"], feh=obj["feh"], isochrone_model="Dotter2016")
WIDTH_COMPARISONS = {
    "Gatto, Marigo2017": (catalog_gatto, regions_gatto, isochrone_marigo, "o", "C1"),
    "Morphology, Marigo2017": (catalog, regions, isochrone_marigo, "s", "C0"),
    "Morphology, Dotter2016": (catalog, regions, isochrone_dotter, "^", "C2"),
}

width_tables = {}
ax = None
for label, (stars, stars_regions, model_isochrone, marker, color) in WIDTH_COMPARISONS.items():
    width_selections = {name: catalog_match_filter_mask(stars, distance_modulus, model_isochrone, {**MATCH_FILTER_SG, **width})
                        for name, width in WIDTHS.items()}
    width_tables[label] = compare_selections(stars_regions, width_selections)
    fig, ax = plot_selection_comparison(width_tables[label], ax=ax, marker=marker, color=color, label=label)
ax.legend(fontsize="small", loc="upper left", bbox_to_anchor=(1.0, 1.0))  # outside the points
display(pd.DataFrame({label: table["snr"] for label, table in width_tables.items()}).round(1))

In [ ]:
# On - off Hess diagram (VIS vs VIS - H, stars passing the error cuts of the match filter) of each star selection,
# with the isochrone (solid) and the match filter (dotted) of each model, and the SNR of each match filter: the
# isochrone of the object (LVDB age, else the default one, and [Fe/H] converted to the Z of each model)
ISOCHRONE_MODEL_COLORS = {"Marigo2017": "k", "Dotter2016": "C2"}
mag_cols = match_filter_cmd_columns(MATCH_FILTER_SG)[0]  # (VIS, H)
LARGER_MF = {**MATCH_FILTER_SG, "color_spread": (0.05, 0.15), "error_multiplier": (3.0, 3.0)}
model_isochrones = {isochrone_model: isochrone_from_feh(obj["age_gyr"], feh=obj["feh"], isochrone_model=isochrone_model)
                    for isochrone_model in ISOCHRONE_MODEL_COLORS}
tracks = {
    isochrone_model: isochrone_cmd_track(distance_modulus, model_isochrone["age"], model_isochrone["metallicity"],
                                         match_filter=LARGER_MF, isochrone_model=isochrone_model)
    for isochrone_model, model_isochrone in model_isochrones.items()
}

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True, layout="constrained")
for ax, (name, stars, stars_regions) in zip(axes, [("Morphology", catalog, regions),
                                                   ("Gatto", catalog_gatto, regions_gatto)]):
    shown = prepare_match_filter_catalog(stars, LARGER_MF, as_mask=True)  # its error cuts
    _, _, difference = on_off_hess(stars, mag_cols, shown & stars_regions["on"], stars_regions["on_area_deg2"],
                                   [shown & stars_regions["off"]], [stars_regions["off_area_deg2"]])
    plot_hess(difference, ax=ax, difference=True, smoothing_bins=1,
              title=f"{name}: on $-$ off, {difference['n_stars']:.0f} excess stars")
    for isochrone_model, color in ISOCHRONE_MODEL_COLORS.items():
        plot_isochrone(ax, tracks[isochrone_model], color=color, label=isochrone_model)
        plot_match_filter_polygon(ax, distance_modulus, model_isochrones[isochrone_model], LARGER_MF,
                                  color=color, ls=":", label=f"{isochrone_model} match filter")
    for row, (isochrone_model, color) in enumerate(ISOCHRONE_MODEL_COLORS.items()):  # top right, one line per model
        selected = catalog_match_filter_mask(stars, distance_modulus, model_isochrones[isochrone_model], LARGER_MF)
        snr = selection_snr(stars_regions, selected)["snr"]
        ax.text(0.97, 0.97 - 0.06 * row, f"SNR = {snr:.0f}", transform=ax.transAxes, ha="right", va="top", color=color)
    ax.legend(loc="upper left", fontsize="small")
axes[1].set_ylabel("")

In [ ]:
MATCH_FILTERS = {
    "VIS-H": {"name": "vis_h", "reference_band": ("euclid_dr1", "VIS"), "color_bands": [("euclid_dr1", "H")]},
    "VIS-J": {"name": "vis_j", "reference_band": ("euclid_dr1", "VIS"), "color_bands": [("euclid_dr1", "J")]},
    "VIS-Y": {"name": "vis_y", "reference_band": ("euclid_dr1", "VIS"), "color_bands": [("euclid_dr1", "Y")]},
    "VIS-Y,J,H": {"name": "vis_yjh", "reference_band": ("euclid_dr1", "VIS"),
                  "color_bands": [("euclid_dr1", "Y"), ("euclid_dr1", "J"), ("euclid_dr1", "H")]},
}
selections = {name: catalog_match_filter_mask(catalog_gatto, distance_modulus, isochrone, match_filter)
              for name, match_filter in MATCH_FILTERS.items()}
# The other isochrone model, same [Fe/H] and age
for isochrone_model in ("Marigo2017", "Dotter2016"):
    other = isochrone_from_feh(feh=obj["feh"], isochrone_model=isochrone_model)
    selections[f"VIS-H, {isochrone_model}"] = catalog_match_filter_mask(catalog_gatto, distance_modulus, other, MATCH_FILTERS["VIS-H"])
table = compare_selections(regions_gatto, selections)
display(table.round(2))
fig, ax = plot_selection_comparison(table)

## 3. Match filters: several band combinations

In [ ]:
MATCH_FILTERS = {
    "VIS-H": {"name": "vis_h", "reference_band": ("euclid_dr1", "VIS"), "color_bands": [("euclid_dr1", "H")]},
    "VIS-J": {"name": "vis_j", "reference_band": ("euclid_dr1", "VIS"), "color_bands": [("euclid_dr1", "J")]},
    "VIS-Y": {"name": "vis_y", "reference_band": ("euclid_dr1", "VIS"), "color_bands": [("euclid_dr1", "Y")]},
    "VIS-Y,J,H": {"name": "vis_yjh", "reference_band": ("euclid_dr1", "VIS"),
                  "color_bands": [("euclid_dr1", "Y"), ("euclid_dr1", "J"), ("euclid_dr1", "H")]},
}
selections = {name: catalog_match_filter_mask(catalog, distance_modulus, isochrone, match_filter)
              for name, match_filter in MATCH_FILTERS.items()}
# The other isochrone model, same [Fe/H] and age
for isochrone_model in ("Marigo2017", "Dotter2016"):
    other = isochrone_from_feh(feh=obj["feh"], isochrone_model=isochrone_model)
    selections[f"VIS-H, {isochrone_model}"] = catalog_match_filter_mask(catalog, distance_modulus, other, MATCH_FILTERS["VIS-H"])
table = compare_selections(regions, selections)
display(table.round(2))
fig, ax = plot_selection_comparison(table)

In [ ]:
BEST = "VIS-H"
fig, ax = plot_object_map(regions, selections[BEST], smoothing_deg=0.03)

## 4. CMDs and Hess diagrams

One panel (CMD) or one row (Hess: on, off, on - off) per colour of the match filter. `stars`: the stars drawn,
here those passing the error cuts of the match filter.

In [ ]:
match_filter = MATCH_FILTERS["VIS-Y,J,H"]
#match_filter = MATCH_FILTERS["VIS-Y"]
shown = prepare_match_filter_catalog(catalog, match_filter, as_mask=True)  # its error cuts
fig, axes = plot_object_cmd(catalog, regions, distance_modulus, isochrone, match_filter, stars=shown)

In [ ]:
fig, axes = plot_object_hess(catalog, regions, distance_modulus, isochrone, match_filter, stars=shown)

## 5. Your own cuts at the catalog level

Any expression on the catalog columns, as a boolean array, combined with a match filter (`mask=`) or with
another selection (`&`).

In [ ]:
vis, h = catalog["euclid_dr1_VIS_obs"].to_numpy(), catalog["euclid_dr1_H_obs"].to_numpy()
my_selections = {
    "VIS-H": selections["VIS-H"],
    "VIS-H, VIS < 24": selections["VIS-H"] & (vis < 24.0),
    "VIS-H, VIS < 25": selections["VIS-H"] & (vis < 25.0),
    "VIS-H, errors < 0.1": catalog_match_filter_mask(
        catalog, distance_modulus, isochrone, MATCH_FILTERS["VIS-H"],
        cuts={"euclid_dr1_VIS_err": (None, 0.1), "euclid_dr1_H_err": (None, 0.1)},
    ),
    "colour box only": (vis - h > -0.5) & (vis - h < 0.3) & (vis < 25.0),
}
my_table = compare_selections(regions, my_selections)
display(my_table.round(2))
fig, ax = plot_selection_comparison(my_table)

In [ ]:
# CMDs and Hess diagrams of any band pairs (colour = first - second), without a match filter
pairs = [("euclid_dr1_Y_obs", "euclid_dr1_H_obs"), ("euclid_dr1_J_obs", "euclid_dr1_H_obs")]
fig, axes = plot_object_cmd(catalog, regions, mag_cols=pairs)
fig, axes = plot_object_hess(catalog, regions, mag_cols=pairs, color_bins=np.arange(-0.5, 1.0, 0.04),
                             mag_bins=np.arange(18.0, 26.0, 1 / 6))